# 00 — Overview

Welcome to **eucare**, a Python library for geometric tilings, Conway operators, and origami crease patterns.

This notebook walks through the *whole* origami pipeline in a few cells:

1. **Build a tiling** — pick a geometry (Euclidean / hyperbolic / spherical) and grow a half-edge graph from prototiles.
2. **Generate a crease pattern (CP)** — apply a CP algorithm such as *shrink-rotate* (SRG) on top of the tiling.
3. **Check flat-foldability and fold** — solve an ILP for the face stacking order to obtain a valid folded state.
4. **Export** — SVG for a plotter, FOLD format for simulators, STL for 3D.

Throughout the series we visualise graphs with the library's built-in Cairo renderer via `G.show(...)` and, for side-by-side panels, `rendering.multi_show([...], titles=[...])`.

The remaining notebooks (`01`–`08`) zoom into each step and add two topical chapters on styling and ad-hoc graph surgery.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


## Step 1: build a tiling

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
G.show(face_inset=0.05, render_vertices=False)


## Step 2: build a crease pattern via shrink-rotate

We render the resulting CP with the standard `rendering.CREASE_PATTERN_PRESET` (no face fill, no vertex markers). The mountain/valley colours are already baked into the SRG's `color_key` attribute by `shrink_rotate_pattern`, so the renderer picks them up automatically.

In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, shrink_rotate_pattern


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = shrink_rotate_pattern(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
SRG = srg_pipeline(G)
SRG.show(**rendering.CREASE_PATTERN_PRESET)


## What's a half-edge graph?

Every tiling and crease pattern in eucare is a **half-edge graph** (DCEL). Each undirected edge is split into two half-edges that link back to each other (`rev`), to the next half-edge around their face (`nex`), and to their origin and destination vertices.

This structure makes it cheap to walk faces, find neighbours, and perform the local surgery that Conway operators and SRG need.

In [ ]:
H = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
H.recompute_lengths_and_angles()
print(f'{len(H.vertices)} vertices, {len(H.halfedges)} half-edges, {len(H.faces)} faces')
H.show(face_inset=0.05, render_vertices=False)


## Where to next

- [`01_Tilings_Euclidean`](01_Tilings_Euclidean.ipynb) — building tilings, Archimedean gallery, Conway operators.
- [`02_Curved_Geometries`](02_Curved_Geometries.ipynb) — spherical and hyperbolic tilings.
- [`03_Crease_Patterns_SRG`](03_Crease_Patterns_SRG.ipynb) — tiling → crease pattern via shrink-rotate.
- [`04_Folding_and_Overlap`](04_Folding_and_Overlap.ipynb) — flat-foldability and the folded state.
- [`05_Conway_Plus_SRG`](05_Conway_Plus_SRG.ipynb) — recipe book.
- [`06_Export_and_3D`](06_Export_and_3D.ipynb) — SVG / FOLD / STL.
- [`07_Styling`](07_Styling.ipynb) — colour and line-width control.
- [`08_Modifications`](08_Modifications.ipynb) — surgery beyond Conway.
